In [1]:
from google.colab import files
uploaded = files.upload()

Saving insurance.csv to insurance.csv


In [2]:
import pandas as pd
import sqlite3

# Cargar CSV
df = pd.read_csv('insurance.csv')

# Crear base de datos en memoria
conn = sqlite3.connect('insurance.db')
df.to_sql('insurance', conn, if_exists='replace', index=False)

print(f"✓ Tabla creada con {len(df)} filas y {len(df.columns)} columnas")
print(f"✓ Columnas: {list(df.columns)}")

✓ Tabla creada con 1338 filas y 7 columnas
✓ Columnas: ['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']


In [3]:
query = "SELECT * FROM insurance LIMIT 5"
pd.read_sql(query, conn)

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [4]:
# Celda 4 — Prima promedio por grupo de riesgo (edad + fumador)
query = """
SELECT
  CASE
    WHEN age < 30 THEN 'Young (< 30)'
    WHEN age BETWEEN 30 AND 50 THEN 'Mid (30-50)'
    ELSE 'Senior (> 50)'
  END AS age_group,
  smoker,
  ROUND(AVG(charges), 2) AS avg_charge,
  COUNT(*) AS n_people
FROM insurance
GROUP BY age_group, smoker
ORDER BY avg_charge DESC
"""
pd.read_sql(query, conn)

,age_group,smoker,avg_charge,n_people
0,Senior (> 50),yes,38820.22,64
1,Mid (30-50),yes,31699.34,124
2,Young (< 30),yes,27518.04,86
3,Senior (> 50),no,13540.28,292
4,Mid (30-50),no,8067.47,441
5,Young (< 30),no,4418.57,331


In [5]:
# Celda 5 — Costo promedio por categoría de BMI
query = """
SELECT
  CASE
    WHEN bmi < 18.5 THEN '1. Underweight'
    WHEN bmi BETWEEN 18.5 AND 24.9 THEN '2. Normal'
    WHEN bmi BETWEEN 25 AND 29.9 THEN '3. Overweight'
    ELSE '4. Obese'
  END AS bmi_category,
  ROUND(AVG(charges), 2) AS avg_charge,
  ROUND(MIN(charges), 2) AS min_charge,
  ROUND(MAX(charges), 2) AS max_charge,
  COUNT(*) AS n_people
FROM insurance
GROUP BY bmi_category
ORDER BY bmi_category
"""
pd.read_sql(query, conn)

,bmi_category,avg_charge,min_charge,max_charge,n_people
0,1. Underweight,8852.20,1621.34,32734.19,20
1,2. Normal,10379.50,1121.87,35069.37,222
2,3. Overweight,10993.99,1252.41,38245.59,377
3,4. Obese,15479.55,1131.51,63770.43,719


In [6]:
# Celda 6 — Perfil de alto riesgo: obeso + fumador
query = """
SELECT
  smoker,
  CASE
    WHEN bmi >= 30 THEN 'Obese'
    ELSE 'Non-obese'
  END AS bmi_group,
  ROUND(AVG(charges), 2) AS avg_charge,
  ROUND(MAX(charges), 2) AS max_charge,
  COUNT(*) AS n_people
FROM insurance
GROUP BY smoker, bmi_group
ORDER BY avg_charge DESC
"""
pd.read_sql(query, conn)

,smoker,bmi_group,avg_charge,max_charge,n_people
0,yes,Obese,41557.99,63770.43,145
1,yes,Non-obese,21363.22,38245.59,129
2,no,Obese,8842.69,36910.61,562
3,no,Non-obese,7977.03,35160.13,502


In [7]:
# Celda 7 — Costo promedio por región
query = """
SELECT
  region,
  ROUND(AVG(charges), 2) AS avg_charge,
  ROUND(AVG(bmi), 2) AS avg_bmi,
  ROUND(100.0 * SUM(CASE WHEN smoker = 'yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_smokers,
  COUNT(*) AS n_people
FROM insurance
GROUP BY region
ORDER BY avg_charge DESC
"""
pd.read_sql(query, conn)

,region,avg_charge,avg_bmi,pct_smokers,n_people
0,southeast,14735.41,33.36,25.0,364
1,northeast,13406.38,29.17,20.7,324
2,northwest,12417.58,29.20,17.8,325
3,southwest,12346.94,30.60,17.8,325
